# Analytical measurements: two-dimensional transient heat conduction

This notebook reproduces the Peacemanâ€“Rachford alternating-direction implicit (ADI) solution described by N. Abdul Halif and N. Rusli, *ASM Science Journal* 12(SI6), 2019, pp. 28â€“33, and compares it with the analytical Fourier-series solution.

**Array convention:** throughout, `T[j, i]` denotes temperature at $x_i,y_j$; `i` indexes $x$ and `j` indexes $y$.

The paper does not explicitly report its time step. The baseline therefore uses the user-editable assumption $\Delta t=0.06$ hr; results are not fitted to the published values.

## 1â€“2. Problem definition, governing equation, and boundary conditions

We solve
$$\frac{\partial T}{\partial t}=\alpha\left(\frac{\partial^2T}{\partial x^2}+\frac{\partial^2T}{\partial y^2}\right),\qquad 0\le x,y\le3\ \mathrm{m},$$
with $\alpha=k/(\rho c_p)=1.25\ \mathrm{m^2/hr}$, interior initial temperature $T_0=30^\circ$F, and zero Dirichlet temperature on all four edges. Boundary values are imposed immediately at $t=0$ and after every half-step.

## 3. Peacemanâ€“Rachford ADI derivation

Let $r_x=\alpha\Delta t/(2\Delta x^2)$ and $r_y=\alpha\Delta t/(2\Delta y^2)$. The first half-step is implicit in $x$:
$$-r_xT_{i-1,j}^{n+1/2}+(1+2r_x)T_{i,j}^{n+1/2}-r_xT_{i+1,j}^{n+1/2}
=r_yT_{i,j-1}^{n}+(1-2r_y)T_{i,j}^{n}+r_yT_{i,j+1}^{n}.$$
The second is implicit in $y$:
$$-r_yT_{i,j-1}^{n+1}+(1+2r_y)T_{i,j}^{n+1}-r_yT_{i,j+1}^{n+1}
=r_xT_{i-1,j}^{n+1/2}+(1-2r_x)T_{i,j}^{n+1/2}+r_xT_{i+1,j}^{n+1/2}.$$
Each line is solved with the Thomas tridiagonal algorithm; no full 2-D matrix is formed or inverted.

## 4. Parameter definitions

In [ ]:
from pathlib import Path
from time import perf_counter
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

Lx = Ly = 3.0
rho = cp = 1.0
k = 1.25
alpha = k / (rho * cp)
T0 = 30.0
dx = dy = 0.3
dt = 0.06                 # assumed because the paper does not report dt clearly
t_final = 1.2
n_terms = 101             # maximum Fourier mode number (odd modes only)
OUTPUT_DIR = Path("analytical_measurements_output")
OUTPUT_DIR.mkdir(exist_ok=True)

rx = alpha * dt / (2 * dx**2)
ry = alpha * dt / (2 * dy**2)
print(f"r_x = {rx:.12g}; r_y = {ry:.12g}")
print("Paper time step not explicitly reported; dt = 0.06 hr is the assumed default.")

## 5. Tridiagonal solver

In [ ]:
def thomas_solve(lower: np.ndarray, diagonal: np.ndarray,
                 upper: np.ndarray, rhs: np.ndarray) -> np.ndarray:
    '''Solve a tridiagonal linear system with the Thomas algorithm.'''
    a, b, c, d = map(lambda z: np.asarray(z, dtype=float).copy(),
                     (lower, diagonal, upper, rhs))
    n = b.size
    if d.size != n or a.size != max(0, n - 1) or c.size != max(0, n - 1):
        raise ValueError("Incompatible tridiagonal-array lengths.")
    if n == 0:
        return np.empty(0)
    for i in range(1, n):
        if abs(b[i - 1]) < np.finfo(float).eps:
            raise np.linalg.LinAlgError("Zero pivot in Thomas algorithm.")
        factor = a[i - 1] / b[i - 1]
        b[i] -= factor * c[i - 1]
        d[i] -= factor * d[i - 1]
    if abs(b[-1]) < np.finfo(float).eps:
        raise np.linalg.LinAlgError("Zero pivot in Thomas algorithm.")
    x = np.empty(n)
    x[-1] = d[-1] / b[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]
    return x

## 6. ADI implementation

In [ ]:
def apply_boundary_conditions(T: np.ndarray, boundary_value: float = 0.0) -> np.ndarray:
    '''Set all four edges in-place and return T.'''
    T[0, :] = T[-1, :] = boundary_value
    T[:, 0] = T[:, -1] = boundary_value
    return T


def _validate_grid(L: float, h: float, name: str) -> int:
    if L <= 0 or h <= 0:
        raise ValueError(f"{name} length and spacing must be positive.")
    intervals = L / h
    if not np.isclose(intervals, round(intervals), rtol=0, atol=1e-10):
        raise ValueError(f"{name} spacing must divide its domain length exactly.")
    return int(round(intervals)) + 1


def adi_half_step_x(T: np.ndarray, alpha: float, dt: float,
                    dx: float, dy: float) -> np.ndarray:
    '''First PR half-step: implicit along x, explicit along y.'''
    ny, nx = T.shape
    rx, ry = alpha * dt / (2 * dx**2), alpha * dt / (2 * dy**2)
    out = np.zeros_like(T, dtype=float)
    n = nx - 2
    lower = np.full(n - 1, -rx); diagonal = np.full(n, 1 + 2 * rx); upper = np.full(n - 1, -rx)
    for j in range(1, ny - 1):
        rhs = ry*T[j-1, 1:-1] + (1-2*ry)*T[j, 1:-1] + ry*T[j+1, 1:-1]
        rhs = rhs.copy()
        rhs[0] += rx * out[j, 0]       # known left boundary contribution
        rhs[-1] += rx * out[j, -1]     # known right boundary contribution
        out[j, 1:-1] = thomas_solve(lower, diagonal, upper, rhs)
    return apply_boundary_conditions(out)


def adi_half_step_y(T_half: np.ndarray, alpha: float, dt: float,
                    dx: float, dy: float) -> np.ndarray:
    '''Second PR half-step: implicit along y, explicit along x.'''
    ny, nx = T_half.shape
    rx, ry = alpha * dt / (2 * dx**2), alpha * dt / (2 * dy**2)
    out = np.zeros_like(T_half, dtype=float)
    n = ny - 2
    lower = np.full(n - 1, -ry); diagonal = np.full(n, 1 + 2 * ry); upper = np.full(n - 1, -ry)
    for i in range(1, nx - 1):
        rhs = rx*T_half[1:-1, i-1] + (1-2*rx)*T_half[1:-1, i] + rx*T_half[1:-1, i+1]
        rhs = rhs.copy()
        rhs[0] += ry * out[0, i]       # known bottom boundary contribution
        rhs[-1] += ry * out[-1, i]     # known top boundary contribution
        out[1:-1, i] = thomas_solve(lower, diagonal, upper, rhs)
    return apply_boundary_conditions(out)


def solve_adi(Lx: float, Ly: float, dx: float, dy: float, alpha: float,
              T0: float, dt: float, t_final: float):
    '''Integrate to t_final and return coordinates, field, and histories.'''
    if alpha <= 0 or dt <= 0 or t_final < 0 or T0 < 0:
        raise ValueError("Require alpha, dt > 0 and t_final, T0 >= 0.")
    nx, ny = _validate_grid(Lx, dx, "x"), _validate_grid(Ly, dy, "y")
    x, y = np.linspace(0, Lx, nx), np.linspace(0, Ly, ny)
    T = np.full((ny, nx), float(T0)); apply_boundary_conditions(T)
    times, centers, maxima = [0.0], [T[ny//2, nx//2]], [T.max()]
    ratio = t_final / dt if dt else 0
    if np.isclose(ratio, round(ratio), rtol=0, atol=1e-10):
        steps = [dt] * int(round(ratio))
    else:
        n_full = int(np.floor(ratio)); steps = [dt] * n_full
        remainder = t_final - n_full * dt
        if remainder > 10*np.finfo(float).eps * max(1.0, t_final):
            steps.append(remainder)
            warnings.warn(f"Final ADI step safely shortened to {remainder:g} hr.")
    elapsed = 0.0
    for step_dt in steps:
        T_half = adi_half_step_x(T, alpha, step_dt, dx, dy)
        T = adi_half_step_y(T_half, alpha, step_dt, dx, dy)
        elapsed += step_dt
        times.append(elapsed); centers.append(T[ny//2, nx//2]); maxima.append(T.max())
    return x, y, T, np.array(times), np.array(centers), np.array(maxima)

## 7. Analytical Fourier-series solution

In [ ]:
def analytical_temperature(x, y, t, alpha, Lx, Ly, T0, n_terms=101):
    '''Evaluate the double sine series; n_terms is the maximum mode number.'''
    if n_terms < 1 or alpha <= 0 or Lx <= 0 or Ly <= 0 or t < 0:
        raise ValueError("Invalid analytical-solution parameters.")
    modes = np.arange(1, int(n_terms) + 1, 2, dtype=float)
    mx, ny = np.meshgrid(modes, modes, indexing="ij")
    terms = (np.sin(mx*np.pi*x/Lx) * np.sin(ny*np.pi*y/Ly) / (mx*ny)
             * np.exp(-alpha*np.pi**2*t*(mx**2/Lx**2 + ny**2/Ly**2)))
    return float(16*T0/np.pi**2 * terms.sum())


def analytical_field(x, y, t, alpha, Lx, Ly, T0, n_terms=101):
    '''Evaluate the analytical solution on a Cartesian grid as T[j,i].'''
    field = np.empty((len(y), len(x)), dtype=float)
    for j, yj in enumerate(y):
        for i, xi in enumerate(x):
            field[j, i] = analytical_temperature(xi, yj, t, alpha, Lx, Ly, T0, n_terms)
    apply_boundary_conditions(field)
    return field

## 8. Baseline simulation and automatic validation

In [ ]:
x, y, T_adi, times, center_history, max_history = solve_adi(Lx, Ly, dx, dy, alpha, T0, dt, t_final)
T_exact = analytical_field(x, y, t_final, alpha, Lx, Ly, T0, n_terms)

def validate_solution(T, T_exact, center_history, max_history, atol=1e-10):
    boundaries = np.r_[T[0, :], T[-1, :], T[:, 0], T[:, -1]]
    exact_boundaries = np.r_[T_exact[0, :], T_exact[-1, :], T_exact[:, 0], T_exact[:, -1]]
    checks = {
        "ADI boundaries are zero": np.allclose(boundaries, 0, atol=atol),
        "ADI is nonnegative": T.min() >= -atol,
        "maximum is nonincreasing": np.all(np.diff(max_history) <= atol),
        "x symmetry": np.allclose(T, T[:, ::-1], atol=atol),
        "y symmetry": np.allclose(T, T[::-1, :], atol=atol),
        "all values finite": np.isfinite(T).all(),
        "center is nonincreasing": np.all(np.diff(center_history) <= atol),
        "analytical boundaries are zero": np.allclose(exact_boundaries, 0, atol=atol),
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise AssertionError("Validation failed: " + ", ".join(failed))
    return pd.Series(checks, name="passed")

display(validate_solution(T_adi, T_exact, center_history, max_history).to_frame())
ci, cj = len(x)//2, len(y)//2
print(f"Recreated ADI center at 1.2 hr: {T_adi[cj, ci]:.9f} Â°F (published â‰ˆ 1.827 Â°F)")
print(f"Analytical center at 1.2 hr:    {T_exact[cj, ci]:.9f} Â°F (published â‰ˆ 1.812 Â°F)")

## 9. Comparison with published results

In [ ]:
published_x = 0.3 * np.arange(1, 10, dtype=float)
published_adi = np.array([0.175, 0.332, 0.457, 0.537, 0.565, 0.537, 0.457, 0.332, 0.174])
published_exact = np.array([0.173, 0.329, 0.453, 0.533, 0.560, 0.533, 0.453, 0.329, 0.173])
j03 = int(round(0.3/dy)); indices = np.rint(published_x/dx).astype(int)
recreated_adi, recreated_exact = T_adi[j03, indices], T_exact[j03, indices]
comparison = pd.DataFrame({
    "x_m": published_x, "published_ADI_F": published_adi, "recreated_ADI_F": recreated_adi,
    "published_analytical_F": published_exact, "recreated_analytical_F": recreated_exact,
    "abs_difference_published_ADI_F": np.abs(recreated_adi-published_adi),
    "abs_difference_published_analytical_F": np.abs(recreated_exact-published_exact),
    "recreated_ADI_vs_analytical_percent_error": 100*np.abs(recreated_adi-recreated_exact)/np.abs(recreated_exact),
})
display(comparison)

## 10. Error analysis

In [ ]:
def error_metrics(numerical, exact):
    e = numerical[1:-1, 1:-1] - exact[1:-1, 1:-1]
    ref = exact[1:-1, 1:-1]
    cy, cx = numerical.shape[0]//2, numerical.shape[1]//2
    center_abs = abs(numerical[cy, cx] - exact[cy, cx])
    return {
        "maximum absolute error (Â°F)": np.max(np.abs(e)),
        "mean absolute error (Â°F)": np.mean(np.abs(e)),
        "RMSE (Â°F)": np.sqrt(np.mean(e**2)),
        "relative L2 error": np.linalg.norm(e.ravel())/np.linalg.norm(ref.ravel()),
        "center absolute error (Â°F)": center_abs,
        "center percent error": 100*center_abs/abs(exact[cy, cx]),
    }

metrics = pd.Series(error_metrics(T_adi, T_exact), name="value")
display(metrics.to_frame())
absolute_error = np.abs(T_adi - T_exact)

## 11. Time-step convergence

In [ ]:
time_rows = []
for dt_i in [0.12, 0.06, 0.03, 0.015]:
    tic = perf_counter()
    x_i, y_i, T_i, *_ = solve_adi(Lx, Ly, dx, dy, alpha, T0, dt_i, t_final)
    runtime = perf_counter() - tic
    exact_i = analytical_field(x_i, y_i, t_final, alpha, Lx, Ly, T0, n_terms)
    m = error_metrics(T_i, exact_i)
    time_rows.append({"dt_hr": dt_i, "center_temperature_F": T_i[len(y_i)//2, len(x_i)//2],
                      "center_error_F": m["center absolute error (Â°F)"],
                      "maximum_absolute_field_error_F": m["maximum absolute error (Â°F)"],
                      "RMSE_F": m["RMSE (Â°F)"], "relative_L2_error": m["relative L2 error"],
                      "runtime_s": runtime})
time_convergence = pd.DataFrame(time_rows)
for col in ["center_error_F", "maximum_absolute_field_error_F", "RMSE_F"]:
    slope = np.polyfit(np.log(time_convergence.dt_hr), np.log(time_convergence[col]), 1)[0]
    time_convergence.attrs[f"observed_order_{col}"] = slope
    print(f"Observed temporal slope for {col}: {slope:.3f}")
display(time_convergence)

## 12. Spatial convergence

In [ ]:
space_rows = []
for h in [0.6, 0.3, 0.15, 0.075]:
    dt_h = 0.00375  # small common time step; divides 1.2 hr exactly
    tic = perf_counter()
    x_h, y_h, T_h, *_ = solve_adi(Lx, Ly, h, h, alpha, T0, dt_h, t_final)
    runtime = perf_counter() - tic
    exact_h = analytical_field(x_h, y_h, t_final, alpha, Lx, Ly, T0, n_terms)
    m = error_metrics(T_h, exact_h)
    space_rows.append({"dx_dy_m": h, "nodes": len(x_h)*len(y_h),
                       "nodes_per_direction": len(x_h), "center_temperature_F": T_h[len(y_h)//2, len(x_h)//2],
                       "center_error_F": m["center absolute error (Â°F)"],
                       "maximum_absolute_field_error_F": m["maximum absolute error (Â°F)"],
                       "RMSE_F": m["RMSE (Â°F)"], "relative_L2_error": m["relative L2 error"],
                       "runtime_s": runtime})
spatial_convergence = pd.DataFrame(space_rows)
for col in ["center_error_F", "maximum_absolute_field_error_F", "RMSE_F"]:
    slope = np.polyfit(np.log(spatial_convergence.dx_dy_m), np.log(spatial_convergence[col]), 1)[0]
    spatial_convergence.attrs[f"observed_order_{col}"] = slope
    print(f"Observed spatial slope for {col}: {slope:.3f}")
display(spatial_convergence)

## 13. Plots (each plot is a separate figure)

In [ ]:
X, Y = np.meshgrid(x, y)

def surface_plot(Z, title, zlabel, cmap="viridis"):
    fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(X, Y, Z, cmap=cmap); fig.colorbar(surf, ax=ax, shrink=.65, label=zlabel)
    ax.set(xlabel="x (m)", ylabel="y (m)", zlabel=zlabel, title=title); plt.show()

def contour_plot(Z, title, label, cmap="viridis"):
    plt.figure(figsize=(7, 6)); c = plt.contourf(X, Y, Z, levels=30, cmap=cmap)
    plt.colorbar(c, label=label); plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.title(title)
    plt.axis("equal"); plt.tight_layout(); plt.show()

surface_plot(T_adi, "ADI temperature at t = 1.2 hr", "Temperature (Â°F)")
surface_plot(T_exact, "Analytical temperature at t = 1.2 hr", "Temperature (Â°F)")
surface_plot(absolute_error, "Absolute error at t = 1.2 hr", "Absolute error (Â°F)", "magma")
contour_plot(T_adi, "ADI filled contour at t = 1.2 hr", "Temperature (Â°F)")
contour_plot(T_exact, "Analytical filled contour at t = 1.2 hr", "Temperature (Â°F)")
contour_plot(absolute_error, "Absolute-error contour at t = 1.2 hr", "Absolute error (Â°F)", "magma")

plt.figure(figsize=(8, 5)); plt.plot(x, T_adi[j03], "o-", label="Recreated ADI")
plt.plot(x, T_exact[j03], "-", label="Recreated analytical")
plt.scatter(published_x, published_adi, marker="x", s=60, label="Published ADI")
plt.scatter(published_x, published_exact, facecolors="none", edgecolors="k", label="Published analytical")
plt.xlabel("x (m)"); plt.ylabel("Temperature (Â°F)"); plt.title("Temperature at y = 0.3 m, t = 1.2 hr")
plt.grid(True, alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 5)); plt.plot(times, center_history, "o-"); plt.xlabel("Time (hr)")
plt.ylabel("Center temperature (Â°F)"); plt.title("Center temperature versus time"); plt.grid(True); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 5)); plt.plot(times, max_history, "o-"); plt.xlabel("Time (hr)")
plt.ylabel("Maximum temperature (Â°F)"); plt.title("Maximum temperature versus time"); plt.grid(True); plt.tight_layout(); plt.show()

mode_limits = np.arange(1, 102, 2)
center_by_modes = [analytical_temperature(Lx/2, Ly/2, t_final, alpha, Lx, Ly, T0, q) for q in mode_limits]
plt.figure(figsize=(8, 5)); plt.semilogx(mode_limits, center_by_modes, "o-"); plt.xlabel("Maximum Fourier mode number")
plt.ylabel("Analytical center temperature (Â°F)"); plt.title("Fourier-series convergence at the center")
plt.grid(True, which="both"); plt.tight_layout(); plt.show()

for col, label, title in [("center_error_F", "Center error (Â°F)", "Center-point error versus time step"),
                          ("maximum_absolute_field_error_F", "Maximum field error (Â°F)", "Maximum field error versus time step"),
                          ("RMSE_F", "RMSE (Â°F)", "RMSE versus time step")]:
    plt.figure(figsize=(7, 5)); plt.loglog(time_convergence.dt_hr, time_convergence[col], "o-")
    plt.xlabel("Î”t (hr)"); plt.ylabel(label); plt.title(title); plt.grid(True, which="both"); plt.tight_layout(); plt.show()

## 14. CSV export

In [ ]:
def field_dataframe(x, y, field):
    Xg, Yg = np.meshgrid(x, y)
    return pd.DataFrame({"x": Xg.ravel(), "y": Yg.ravel(), "temperature": field.ravel()})

field_dataframe(x, y, T_adi).to_csv(OUTPUT_DIR/"adi_temperature_t1p2.csv", index=False)
field_dataframe(x, y, T_exact).to_csv(OUTPUT_DIR/"analytical_temperature_t1p2.csv", index=False)
field_dataframe(x, y, absolute_error).to_csv(OUTPUT_DIR/"absolute_error_t1p2.csv", index=False)
comparison.to_csv(OUTPUT_DIR/"published_comparison.csv", index=False)
time_convergence.to_csv(OUTPUT_DIR/"time_step_convergence.csv", index=False)
spatial_convergence.to_csv(OUTPUT_DIR/"spatial_convergence.csv", index=False)
pd.DataFrame({"time_hr": times, "center_temperature_F": center_history}).to_csv(
    OUTPUT_DIR/"center_temperature_history.csv", index=False)
print("Exported:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", path)

## 15. Conclusions

The tables and plots above quantify agreement rather than forcing a match. The recreated ADI field should closely follow the analytical solution, while residual discrepancies reflect temporal and spatial truncation error. Reducing $\Delta t$ initially reveals the expected second-order Peacemanâ€“Rachford/Crankâ€“Nicolson behavior, but the measured temporal slope can flatten once spatial error dominates. Grid refinement similarly tends toward second-order spatial convergence when temporal error is sufficiently small.

Exact reproduction of every published number is limited because the paper does not explicitly report its time step; the baseline $\Delta t=0.06$ hr is an informed, editable assumption. Compare the reported slopes and convergence tables before concluding whether the asymptotic second-order regime has been reached.